# Lesson 08 · Overfitting & regularization: an MNIST network memorizes 1,000 images, and the ways to fight it

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/08-chong-qua-khop-en.ipynb)

This notebook accompanies [Lesson 08 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/chong-qua-khop/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/08-chong-qua-khop.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
We take the course's 784 → 128 → 64 → 10 network but let it learn from only **1,000** MNIST images, so few that it
can memorize them. We detect overfitting with a validation set, then write the ways to fight it ourselves in numpy:
the L2 penalty (weight decay), dropout, data augmentation and early stopping. Finally we see why “the bigger the
model, the more it overfits” is not always true (double descent).

**What you'll do:**
1. Train the network on 1,000 images for 150 epochs, watch the validation loss climb while the training loss
   approaches 0, and stop early.
2. Shuffle the labels at random and watch the network memorize even meaningless (image, label) pairs.
3. Write the L2 penalty and dropout, and measure what each one does to the weights and to the validation curve.
4. Shift the images at random, then read the results of a sweep over 30 combinations × 3 seeds: choose on the
   validation set, report on the test set.
5. Sweep the size of a random-features model past the “just enough to memorize” point and watch the validation
   error fall a second time.

**Background:** [Lesson 05 · Gradient descent](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/gradient-descent/)
(mini-batch SGD, epochs) and [Lesson 07 · Momentum, Adam & AdamW](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/)
(weight decay in AdamW). The network, softmax + cross-entropy and backpropagation come from
[Lesson 04](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-mat-mat/) and
[Lesson 06](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/).
**Run time:** under a minute on a CPU, including the first MNIST download (~11 MB).

Every number in the notebook is computed by the cells below, and matches the numbers in the demo. The one exception
is the results of the full sweep (91 training runs of 150 epochs), which are loaded from the very file the demo
uses, because rerunning them takes a few minutes; the notebook reruns four of those training runs and checks that
they give the same numbers.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import gzip
import hashlib
import itertools
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”)."""
    return f"{round(float(x), digits) + 0.0:,.{digits}f}".replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 10,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def fmt_axes(fig):
    """Put the tick labels of every axis of the figure in the English format. A log axis gets ticks at 1–2–5, or at
    the powers of 10 if the axis is very long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() == "log":
                lo, hi = sorted(lim)
                mult = (1, 2, 5) if hi / lo < 300 else (1,)
                axis.set_major_locator(
                    FixedLocator([m * 10.0**k for k in range(-6, 6) for m in mult if lo <= m * 10.0**k <= hi])
                )
                axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(tick))


print(num(13007, 0), num(0.1, 1), num(-4.3805, 4), pct(0.993527))

## 2 · The data: only 1,000 images to learn from

MNIST (LeCun et al., 1998): 60,000 training images and 10,000 test images, grayscale 28×28 handwritten digits.
The cell below downloads the 4 original files and checks their MD5 hashes, so everyone who runs it gets exactly the
same data. It also loads the sweep results file that the demo uses (`fetch_json`, with an MD5 check).

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# The results of a long training run (the same file as the demo on the website). build.py replaces COURSE_DATA_URL
# with the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
runs = fetch_json("mnist-overfit.json", "95bbd8b2d0b3e5b27a1cdedbb6e8fd04")  # the demo's sweep results
print("training:", train_img.shape, " test:", test_img.shape)

We split the data as the demo does, into three parts, each with its own job:
- **training:** 1,000 images, exactly 100 per digit, chosen at random (seed 0) from the first 50,000 images of the
  training file. This is the “old exam” the network practices on;
- **validation:** the last 10,000 images of the training file, the “new exam” for detecting overfitting and
  *choosing* a configuration;
- **test:** the 10,000 images of the test file, the “real exam”, read only at the chosen configuration and not used
  to choose anything.

In [ ]:
X_all, y_all = train_img.reshape(-1, 784) / 255.0, train_lab.astype(np.int64)
pick = np.random.default_rng(0)  # the same seed as the demo, so the same 1,000 images
SUB = np.sort(np.concatenate([pick.choice(np.flatnonzero(y_all[:50000] == d), 100, replace=False) for d in range(10)]))
Xtr, ytr = X_all[SUB], y_all[SUB]  # training
Xval, yval = X_all[50000:], y_all[50000:]  # validation: detect overfitting, choose a configuration
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(np.int64)  # test: for reporting only
SHUFFLED = pick.permutation(ytr)  # the same 1,000 labels, randomly reassigned to the images (section 5)
counts = np.bincount(ytr)
print(
    f"training {num(len(ytr), 0)} images ({counts.min()} images per digit, taken from the first {num(50000, 0)} images)",
    end=" · ",
)
print(f"validation {num(len(yval), 0)} · test {num(len(yte), 0)} images")
print(
    f"the demo's sweep file: {len(runs['configs'])} configurations × {len(runs['seeds'])} seeds, {runs['epochs']} epochs each"
)
assert (len(ytr), len(yval), len(yte)) == (runs["nTrain"], runs["nVal"], runs["nTest"]) and (counts == 100).all()

The figure below shows the first 2 training images of each digit, each with its label above it.

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(10, 2.4))
for d in range(10):
    for row, i in enumerate(np.flatnonzero(ytr == d)[:2]):
        axes[row, d].imshow(Xtr[i].reshape(28, 28), cmap="gray_r")
        axes[row, d].set_title(str(d), fontsize=11)
        axes[row, d].axis("off")
plt.show()

## 3 · The course's network, and a training loop with three knobs

The network of Lessons 04–07: $h_1 = \mathrm{ReLU}(xW_1 + b_1)$, $h_2 = \mathrm{ReLU}(h_1W_2 + b_2)$, $z = h_2W_3 + b_3$,
softmax + cross-entropy, He initialization, plain SGD with $\eta = 0.1$, batch size 64. With 1,000 images, each epoch
has only 16 steps.

The `step()` function below is one SGD step with the backpropagation of Lesson 06, plus two knobs that the later
sections explain: `lam` (the L2 penalty, section 6) and `p` (dropout, section 7). The `evaluate()` function measures
the network in **inference mode**: the full network, no dropout, no penalty added. It computes in float32, which is
twice as fast and still accurate enough for 4 digits.
The two functions `dropout_mask()` and `shift_images()` that `step()` and `train()` call are written in sections 7
and 8; until then we only run with `p = 0` and no image shifts, so they are not called yet.

In [ ]:
SIZES, ETA, BATCH, EPOCHS, PATIENCE, SHIFT = [784, 128, 64, 10], 0.1, 64, 150, 10, 2
Xtr32, Xval32, Xte32 = (X.astype(np.float32) for X in (Xtr, Xval, Xte))


def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def logits(params, X):
    """The forward pass in inference mode: the full network, no dropout."""
    z = X
    for k, (W, b) in enumerate(params):
        z = z @ W.astype(np.float32) + b.astype(np.float32)
        z = np.maximum(0, z) if k < 2 else z
    return z


def evaluate(params, X, y):
    """The mean cross-entropy loss and the number of images predicted correctly, in inference mode."""
    z = logits(params, X)
    m = z.max(axis=1, keepdims=True)
    log_sum = m[:, 0] + np.log(np.exp(z - m).sum(axis=1))  # log Σ e^z, numerically stable (Lesson 04)
    return float((log_sum - z[np.arange(len(y)), y]).mean()), int((z.argmax(1) == y).sum())


def step(params, x, y, lam, p, noise):
    """One SGD step on the batch (x, y): forward (with dropout if p > 0), backward (Lesson 06), update (with the L2
    penalty if lam > 0)."""
    (W1, b1), (W2, b2), (W3, b3) = params
    a1 = x @ W1 + b1
    m1 = (a1 > 0) * (dropout_mask(a1.shape, p, noise) if p else 1)  # ReLU's derivative × the dropout mask (section 7)
    h1 = a1 * m1
    a2 = h1 @ W2 + b2
    m2 = (a2 > 0) * (dropout_mask(a2.shape, p, noise) if p else 1)
    h2 = a2 * m2
    dz = softmax(h2 @ W3 + b3)
    dz[np.arange(len(y)), y] -= 1  # ∂L/∂z = ŷ − y (Lesson 04)
    dz /= len(y)
    da2 = (dz @ W3.T) * m2
    da1 = (da2 @ W2.T) * m1
    grads = [(x.T @ da1, da1.sum(0)), (h1.T @ da2, da2.sum(0)), (h2.T @ dz, dz.sum(0))]
    for layer, (gW, gb) in zip(params, grads):
        layer[0] -= ETA * (gW + 2 * lam * layer[0])  # ∂(λ‖W‖²)/∂W = 2λW (section 6); the biases b are not penalized
        layer[1] -= ETA * gb

The `train()` function runs 150 epochs and records, after each epoch, the loss and the number of correct images on
the training set, along with the size of the weights $\lVert W\rVert^2$ (the sum of the squares of every entry of
$W_1, W_2, W_3$). The validation set takes longer (10,000 images), so it is measured after every epoch up to epoch
`dense`, then every `val_every` epochs (5 by default) and at the last epoch. The third knob, `aug`, shifts the images
at random (section 8).

The function also **stops early** as the demo does: it keeps the weights with the lowest validation loss, and records
the stopping epoch once 10 epochs in a row have brought no new record. It still runs on to the end, so that we can see
what would happen without stopping.

As in `reel/mnist_mlp/overfit.py`, the script that makes the demo's data, each seed gives the same initial weights and
the same batch order in every configuration, so the difference between two configurations comes only from the knob.

In [ ]:
def train(lam=0.0, p=0.0, aug=False, labels=None, seed=0, epochs=EPOCHS, dense=0, val_every=5, stop_early=False):
    labels = ytr if labels is None else labels
    rng = np.random.default_rng(seed)  # initial weights and batch order
    noise = np.random.default_rng([seed, 1])  # dropout masks and image shifts
    params = [[rng.normal(0, np.sqrt(2 / a), (a, b)), np.zeros(b)] for a, b in itertools.pairwise(SIZES)]  # He
    log = {"train": [], "w2": [], "val": {}, "best": None, "stop": None}
    for ep in range(epochs + 1):
        if ep:  # one epoch: 16 batches in random order
            order = rng.permutation(len(Xtr))
            for i in range(0, len(order), BATCH):
                idx = order[i : i + BATCH]
                x = shift_images(Xtr[idx], noise) if aug else Xtr[idx]  # data augmentation (section 8)
                step(params, x, labels[idx], lam, p, noise)
        log["train"].append(evaluate(params, Xtr32, labels))
        log["w2"].append(float(sum((W**2).sum() for W, _ in params)))
        if ep <= dense or ep % val_every == 0 or ep == epochs:
            log["val"][ep] = evaluate(params, Xval32, yval)
        if ep <= dense and log["stop"] is None:  # early stopping, with a patience of PATIENCE epochs
            if log["best"] is None or log["val"][ep][0] < log["val"][log["best"]][0]:
                log["best"], log["best_params"] = ep, [[W.copy(), b.copy()] for W, b in params]
            elif ep - log["best"] >= PATIENCE:
                log["stop"] = ep
                if stop_early:
                    break
    log["params"] = params
    return log


def first_perfect(log, n=1000):
    """The first epoch at which the network gets all n training images right (None if it never does)."""
    return next((e for e, (_, ok) in enumerate(log["train"]) if ok == n), None)


def check(log, ref):
    """Compare with the seed-0 curve the demo stored (losses rounded to 4 digits, ‖W‖² to 1 digit; correct counts off
    by ≤ 3)."""
    ep, n = list(log["val"]), len(log["train"])
    assert np.allclose([log["val"][e][0] for e in ep], [ref["valLoss"][e] for e in ep], atol=5e-4)
    assert max(abs(log["val"][e][1] - ref["valOk"][e]) for e in ep) <= 3
    assert np.allclose([v for v, _ in log["train"]], ref["trainLoss"][:n], atol=5e-4)
    assert max(abs(ok - r) for (_, ok), r in zip(log["train"], ref["trainOk"])) <= 3
    assert np.allclose(log["w2"], ref["w2"][:n], atol=0.1), "differs from the demo: check the data"


def config(lam=0, p=0, aug=False):
    """One cell of the demo's sweep table."""
    return next(c for c in runs["configs"] if c["lam"] == lam and c["p"] == p and c["aug"] == aug)


def seed_stats(c, key):
    """Mean, min and max over the demo's 3 seeds (key: "valOk" or "testOk", at the early-stopping weights)."""
    v = [s[key] / 10000 for s in c["seeds"]]
    return np.mean(v), min(v), max(v)


def fmt_range(c, key):
    m, lo, hi = seed_stats(c, key)
    return f"{pct(m)} ({pct(lo)}–{pct(hi)})"


print(
    f"{num(sum(a * b + b for a, b in itertools.pairwise(SIZES)), 0)} parameters for {num(len(Xtr), 0)} training images"
)
print(f"{-(-len(Xtr) // BATCH)} steps per epoch, {EPOCHS} epochs = {num(EPOCHS * -(-len(Xtr) // BATCH), 0)} steps")

## 4 · Memorizing 1,000 images: top marks on the old exam, low marks on the new one

The network has 109,386 parameters and only 1,000 images to learn from: more than 100 parameters per image. We train
it for 150 epochs without any regularization, measure the validation set after every epoch for the first 30 epochs,
and compare with the seed-0 curve that the demo stored.

In [ ]:
base = train(dense=30)
check(base, config())
b_ep = base["best"]
print(f"first gets all 1,000 training images right: epoch {first_perfect(base)}")
for ep in [0, 10, b_ep, 40, 100, 150]:
    (tl, tok), (vl, vok) = base["train"][ep], base["val"][ep]
    print(
        f"epoch {ep:>3}: training loss {num(tl, 4)}, {pct(tok / 1000, 1):>6} correct  ·  validation loss {num(vl, 4)}, {pct(vok / 10000)} correct"
    )
rise = base["val"][150][0] / base["val"][b_ep][0] - 1
print(f"lowest validation loss at epoch {b_ep}; by epoch 150 it has risen by {pct(rise, 0)}")
assert (first_perfect(base), b_ep, base["stop"]) == (26, 20, 30), "differs from the demo: check the data"

The figure below plots two curves against the epoch: the loss (left) and the accuracy (right). Training is the solid
blue line with circles; validation is the dashed orange line with squares. The vertical green line marks the epoch
with the lowest validation loss.

In [ ]:
TRAIN_STYLE = {"color": BLUE, "marker": "o", "ls": "-", "ms": 3, "lw": 1.6}
VAL_STYLE = {"color": ORANGE, "marker": "s", "ls": (0, (6, 3)), "ms": 3, "lw": 1.6}


def val_curve(log, k=0):
    ep = sorted(log["val"])
    return ep, [log["val"][e][k] for e in ep]


fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.4))
ep_all = np.arange(EPOCHS + 1)
a.plot(ep_all, [v for v, _ in base["train"]], markevery=10, label="training (old exam)", **TRAIN_STYLE)
a.plot(*val_curve(base), markevery=5, label="validation (new exam)", **VAL_STYLE)
a.set(ylim=(0, 1), xlabel="epoch", ylabel="loss", title="Loss (epoch 0 ≈ 2.4, off the chart)")
b.plot(ep_all, [ok / 1000 for _, ok in base["train"]], markevery=10, **TRAIN_STYLE)
ep_v, ok_v = val_curve(base, 1)
b.plot(ep_v, np.array(ok_v) / 10000, markevery=5, **VAL_STYLE)
b.set(ylim=(0.8, 1.01), xlabel="epoch", ylabel="accuracy", title="Accuracy")
for ax in (a, b):
    ax.axvline(b_ep, color=GREEN, lw=1.5)
    ax.grid(alpha=0.3, which="both")
a.annotate(f"lowest: epoch {b_ep}", (b_ep, 0.9), xytext=(6, 0), textcoords="offset points", fontsize=9)
a.legend(frameon=False, fontsize=9, loc="upper right")
fmt_axes(fig)
b.yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
plt.tight_layout()
plt.show()

By epoch 26 the network already gets all 1,000 training images right, and the training loss keeps falling, down to
0.0021 at epoch 150. The validation loss, on the other hand, is lowest at epoch 20 and then climbs: the network gets
better at the “old exam” with every epoch, and worse at the “new exam”. That is **overfitting**: the gap between the
two curves keeps widening.

### 🤔 Predict first
From epoch 20 to epoch 150, the validation loss rises by 30%. So is the validation accuracy at epoch 150 lower than,
equal to, or higher than at epoch 20?

In [ ]:
for name, params in [(f"epoch {b_ep}", base["best_params"]), ("epoch 150", base["params"])]:
    p_val = softmax(logits(params, Xval32))
    wrong = p_val.argmax(1) != yval
    loss_i = -np.log(p_val[np.arange(len(yval)), yval])  # the loss of each validation image
    print(
        f"{name:>9}: {pct(1 - wrong.mean())} correct · mean loss on the wrong images {num(loss_i[wrong].mean(), 2)}, "
        f"on the right ones {num(loss_i[~wrong].mean(), 4)} · wrong images with ≥ 99% confidence: {wrong[p_val.max(1) >= 0.99].sum()}"
    )
assert base["val"][150][1] > base["val"][b_ep][1], "differs from the demo: check the data"

The validation accuracy barely changes; it even edges up a little. The network doesn't get *more images* wrong; it
gets them wrong *more confidently*. The mean loss on the images it gets wrong rises from 2.85 to 4.41, and the number
of wrong images on which the network is ≥ 99% confident rises from 21 to 132. Cross-entropy punishes a confident wrong
prediction very heavily (Lesson 04), so only the loss sees this. Overfitting does not necessarily show up in the
accuracy; keep an eye on the validation loss too.

### Early stopping

The cheapest way to fight overfitting is to stop at the right time: keep the weights from the epoch with the lowest
validation loss. `train()` already does that: the last record is at epoch 20, and after 10 epochs without a new record
it records 30 as the stopping epoch. Only now do we read the test set, once, at the chosen weights:

In [ ]:
test_best = evaluate(base["best_params"], Xte32, yte)[1]
test_final = evaluate(base["params"], Xte32, yte)[1]
print(
    f"early stopping: keeps epoch {b_ep}, stops at epoch {base['stop']} (waits {PATIENCE} epochs without a new record)"
)
print(
    f"test: the epoch-{b_ep} weights get {pct(test_best / 10000)} right · after all 150 epochs, {pct(test_final / 10000)}"
)
seeds_test = [s["testOk"] / 10000 for s in config()["seeds"]]
print(f"demo, 3 seeds, weights at early stopping: test accuracy {pct(min(seeds_test))}–{pct(max(seeds_test))}")
assert abs(test_best - config()["testBest"]) <= 3 and abs(test_final - config()["testFinal"]) <= 3, (
    "differs from the demo"
)

Early stopping cuts out 120 epochs of training, but it optimizes the validation *loss*, not the accuracy: here the
epoch-150 weights actually get slightly more test images right (89.57% versus 89.45%). That difference is smaller than
the variation between seeds, so we can't say which epoch is “better”. Early stopping is fast and cheap, but it doesn't
fix the root of the problem: 1,000 images are too few.

## 5 · Memorizing even meaningless labels

Does the network learn the rule, or does it just memorize? Zhang et al. (2016) tested this very directly: shuffle the
labels at random. The images are still the same 1,000 images, but each image's label is reassigned at random
(`SHUFFLED`, in section 2), so no rule connects the images to the labels any more. The only way is to memorize.

### 🤔 Predict first
With shuffled labels, does the network reach 100% accuracy on the training set? If so, does it need more epochs or
fewer than with the true labels (epoch 26)?

In [ ]:
shuf = train(labels=SHUFFLED, epochs=70)
check(shuf, runs["shuffled"])
f_shuf = first_perfect(shuf)
print(
    f"first gets 100% of the training images right: true labels at epoch {first_perfect(base)}, shuffled labels at epoch {f_shuf}",
    end="",
)
print(f" ({num(f_shuf / first_perfect(base), 1)} times slower)")
for ep in [0, 30, 70]:
    print(
        f"shuffled labels, epoch {ep:>2}: training {pct(shuf['train'][ep][1] / 1000, 1):>6} correct · "
        f"validation {pct(shuf['val'][ep][1] / 10000)} correct · validation loss {num(shuf['val'][ep][0], 2)}"
    )
print(
    f"‖W‖² at epoch 70: true labels {num(base['w2'][70], 1)} · shuffled labels {num(shuf['w2'][70], 1)} (at the start {num(base['w2'][0], 1)})"
)
assert f_shuf == 68 and shuf["val"][70][1] < 1500, "differs from the demo: check the data"

The figure below compares the two training runs, epoch by epoch: true labels (blue, solid, circles) and shuffled
labels (orange, dashed, squares). On the left is the accuracy on the training set, on the right on the validation set;
the dotted grey line is the level of random guessing, 10%.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.3), sharey=True)
for log, label, st in [(base, "true labels", TRAIN_STYLE), (shuf, "randomly shuffled labels", VAL_STYLE)]:
    n = min(len(log["train"]), 71)
    a.plot(np.arange(n), [ok / 1000 for _, ok in log["train"][:n]], markevery=5, label=label, **st)
    ep_v, ok_v = val_curve(log, 1)
    keep = [i for i, e in enumerate(ep_v) if e <= 70]
    b.plot([ep_v[i] for i in keep], [ok_v[i] / 10000 for i in keep], markevery=2, **st)
for ax, title in [(a, "Training set (old exam)"), (b, "Validation set (new exam)")]:
    ax.axhline(0.1, color=GREY, ls=":", lw=1.2)
    ax.set(xlabel="epoch", title=title, ylim=(0, 1.03))
    ax.grid(alpha=0.3)
a.set_ylabel("accuracy")
b.annotate("random guessing: 10%", (70, 0.1), xytext=(0, 5), textcoords="offset points", ha="right", fontsize=9)
a.legend(frameon=False, fontsize=9, loc="lower right")
fmt_axes(fig)
plt.tight_layout()
plt.show()

Yes: it first gets all 1,000 meaningless labels right at epoch 68, only about 2.6 times slower than with the true
labels. On new images it stays around the level of random guessing, because there is nothing to generalize. A low
training loss does **not** prove that the network has understood anything.

Zhang et al. found exactly this with large convolutional networks on CIFAR-10 and ImageNet. They concluded that the
number of parameters alone cannot explain why networks generalize: the same network, with the same algorithm, can
learn the rule or memorize, depending on the data; and memorizing random labels takes longer by only “a small
constant factor”. Notice too: memorizing noise takes larger weights, and ‖W‖² grows much faster. That suggests a way
to fight back: penalize large weights.

## 6 · The L2 penalty: pulling the weights toward 0

We add to the loss a penalty proportional to the sum of the squares of the weights:

$$\mathcal{L}_{\text{total}} = \mathcal{L} + \lambda \lVert W\rVert^2, \qquad
\frac{\partial\, \lambda\lVert W\rVert^2}{\partial W} = 2\lambda W$$

As usual, only the matrices $W$ are penalized, not the biases $b$. With plain SGD, one step is

$$W \leftarrow W - \eta\,(\nabla\mathcal{L} + 2\lambda W) = (1 - 2\eta\lambda)\,W - \eta\,\nabla\mathcal{L}$$

That is, before each step every weight is **shrunk** by the factor $1 - 2\eta\lambda$: that is **weight decay**. In
Lesson 07, AdamW writes the shrinking as $1 - \eta\lambda$, so its $\lambda$ corresponds to $2\lambda$ here. The cell
below checks that the two ways of writing it are the same with plain SGD, on a real batch, and computes how much the
shrinking adds up to over 150 epochs.

In [ ]:
lam = 0.003
rng_probe = np.random.default_rng(5)
W0 = [[rng_probe.normal(0, np.sqrt(2 / a), (a, b)), np.zeros(b)] for a, b in itertools.pairwise(SIZES)]


def copy(params):
    return [[W.copy(), b.copy()] for W, b in params]


penalized, plain = copy(W0), copy(W0)
step(penalized, Xtr[:64], ytr[:64], lam, 0, None)  # one step on L + λ‖W‖²
step(plain, Xtr[:64], ytr[:64], 0, 0, None)  # one ordinary SGD step: W − η∇L
gap = max(float(np.abs(Wp - (Wq - 2 * ETA * lam * W)).max()) for (Wp, _), (Wq, _), (W, _) in zip(penalized, plain, W0))
print(f"step with the L2 penalty versus “ordinary step, then shrink by another 2ηλW”: largest difference {gap:.1e}")
n_steps = EPOCHS * -(-len(Xtr) // BATCH)
shrink = (1 - 2 * ETA * lam) ** n_steps
print(f"λ = {num(lam, 3)}: each step multiplies W by 1 − 2ηλ = {num(1 - 2 * ETA * lam, 4)}")
print(
    f"the shrinking alone, after {num(n_steps, 0)} steps: W is {num(shrink, 3)} times, ‖W‖² {num(shrink**2, 3)} times as large"
)
assert gap < 1e-12

The two agree up to rounding error. If there were only the shrinking, after 150 epochs every weight would be 0.237
times its initial value, and ‖W‖² 0.056 times. The data's gradient pulls the other way: weights that really help
reduce the loss are kept, and weights that only serve to memorize a few images are gradually shrunk toward 0. We rerun
150 epochs with $\lambda = 0.003$:

In [ ]:
l2 = train(lam=0.003, dense=40, val_every=10)
check(l2, config(lam=0.003))
for name, log in [("no penalty", base), ("λ = 0.003", l2)]:
    b_ = log["best"]
    print(
        f"{name:>10}: ‖W‖² at epochs 0 · 20 · 150 = {' · '.join(num(log['w2'][e], 1) for e in (0, 20, 150))} · "
        f"100% of the training set from epoch {first_perfect(log)} · best validation loss {num(log['val'][b_][0], 4)} "
        f"(epoch {b_}), epoch 150: {num(log['val'][150][0], 4)}"
    )
assert (first_perfect(l2), l2["best"], l2["stop"]) == (40, 29, 39), "differs from the demo: check the data"
assert l2["val"][150][0] < l2["val"][l2["best"]][0], "differs from the demo: check the data"

Three things change. The weights go the opposite way: without the penalty ‖W‖² *grows* from 405 to 495; with it,
‖W‖² shrinks to 95.1 (still larger than the 405 × 0.056 of the shrinking alone). The network still memorizes all
1,000 images, only more slowly (epoch 40 instead of 26). And the validation loss no longer climbs: at epoch 150 it is
even lower than at epoch 29, the record that early stopping kept. With the L2 penalty, early stopping stops too early.

$\lambda$ is a **hyperparameter**: too weak a penalty has no effect, too strong a penalty keeps the network from
fitting even the training data. The figure below plots all 5 values of $\lambda$ that the demo tried (seed 0, from the
demo's file; the curves for $\lambda = 0$ and $0.003$ coincide with the two runs we just did). Left: ‖W‖² against the
epoch. Right: the validation loss, measured every epoch. Each $\lambda$ has its own color and marker shape.

In [ ]:
LAM_STYLE = {
    0: (GREY, "o", "-"),
    0.0003: (BLUE, "v", (0, (1, 1.5))),
    0.001: (BLUE, "s", (0, (6, 3))),
    0.003: (GREEN, "^", "-"),
    0.01: (ORANGE, "D", (0, (3, 1, 1, 1))),
}
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
for lam_, (color, mk, ls) in LAM_STYLE.items():
    c = config(lam=lam_)
    kw = {"color": color, "marker": mk, "ls": ls, "ms": 4, "lw": 1.5, "markevery": 15}
    a.plot(ep_all, c["w2"], label=f"λ = {tick(lam_)}", **kw)
    b.plot(ep_all, c["valLoss"], **kw)
a.set(xlabel="epoch", ylabel="‖W‖²", title="Size of the weights")
b.set(xlabel="epoch", ylabel="validation loss", ylim=(0.3, 0.55), title="Validation loss")
for ax in (a, b):
    ax.grid(alpha=0.3)
a.legend(frameon=False, fontsize=9, ncol=2)
fmt_axes(fig)
plt.tight_layout()
plt.show()
c01 = config(lam=0.01)
print(
    f"λ = 0.01: {pct(c01['trainOk'][-1] / 1000, 1)} of the training set correct at epoch 150; best validation loss {num(c01['valLoss'][c01['best']], 4)}"
)

The stronger the penalty, the smaller ‖W‖², and the flatter the validation-loss curve at the end. But
$\lambda = 0.01$ (orange, diamonds) goes too far: the weights are squeezed so small that the network no longer gets
the whole training set right, and its best validation loss is higher even than with no penalty. How do we choose
$\lambda$? With the validation set, in section 9.

## 7 · Dropout: a different subnetwork for each image, at each step

Dropout (Hinton et al., 2012; Srivastava et al., 2014) randomly switches off a fraction of the hidden neurons at every
forward pass during training. Each neuron is switched off with probability $p$, independently of the other neurons
and of the other images:

$$\tilde h = \frac{m \odot h}{1-p}, \qquad m_i \sim \operatorname{Bernoulli}(1-p)$$

Dividing by $1 - p$ keeps the expected value unchanged: $\mathbb{E}[\tilde h_i] = (1-p)\cdot\frac{h_i}{1-p} = h_i$.
Thanks to that, at inference time we use the full network without changing anything (this formulation is called
*inverted dropout*, and it is also how PyTorch does it). Note: Srivastava et al. use $p$ for the probability of
**keeping** a neuron, the opposite of here and of PyTorch.

Backpropagating through dropout is simple too: a neuron that is switched off passes no gradient, and the remaining
neurons pass their gradient multiplied by $1/(1-p)$. That is why `step()` merges the dropout mask with the ReLU
derivative into one array, `m1`, `m2`.

In [ ]:
def dropout_mask(shape, p, rng):
    """Inverted-dropout mask: 0 with probability p, otherwise 1/(1 − p). Each row (each image) gets its own mask."""
    return (rng.random(shape) >= p) / (1 - p)


demo_rng = np.random.default_rng(3)
m = dropout_mask((64, 128), 0.2, demo_rng)  # a batch of 64 images, a hidden layer of 128 neurons
print(
    f"fraction of neurons switched off in the batch: {pct((m == 0).mean())} · value of the remaining ones: {num(m.max(), 2)}"
)
print(
    f"mean of the mask: {num(m.mean(), 3)} (its expected value is exactly 1) · distinct masks among the 64 images: {len({r.tobytes() for r in m})}"
)
big = dropout_mask((1000, 128), 0.2, demo_rng)
print(f"over 1,000 × 128 draws: mean {num(big.mean(), 4)}")
assert abs(big.mean() - 1) < 0.01

About 20% of the neurons are switched off, the remaining ones are multiplied by 1.25, and the mean of the mask is
very close to 1. Each image in the batch has its own mask, and `train()` draws new masks at every step. So in 150
epochs the network never sees the same subnetwork twice: each neuron has to be useful whether or not the other
neurons are present, instead of relying on one particular group to memorize an image together (Hinton et al. call
this avoiding “co-adaptation”).

### 🤔 Predict first
The L2 penalty fights overfitting by keeping the weights small. With dropout $p = 0.2$ on both hidden layers (and no
L2 penalty), is ‖W‖² after 150 epochs smaller or larger than without dropout (495)?

In [ ]:
drop = train(p=0.2, dense=40, val_every=10)
check(drop, config(p=0.2))
for name, log in [("no dropout", base), ("dropout 0.2", drop)]:
    b_ = log["best"]
    print(
        f"{name:>11}: ‖W‖² at epoch 150 = {num(log['w2'][150], 1)} · 100% of the training set from epoch {first_perfect(log)} · "
        f"best validation loss {num(log['val'][b_][0], 4)} (epoch {b_}) → epoch 150: {num(log['val'][150][0], 4)}"
    )
assert (first_perfect(drop), drop["best"], drop["stop"]) == (38, 28, 38), "differs from the demo: check the data"
print(f"demo, 3 seeds, validation accuracy at early stopping: no dropout {fmt_range(config(), 'valOk')}", end=" · ")
print(f"dropout 0.2 {fmt_range(config(p=0.2), 'valOk')}")
assert drop["w2"][150] > base["w2"][150], "differs from the demo: check the data"

Larger: dropout doesn't make the weights smaller; if anything, the opposite. It fights overfitting in a different way:
memorizing gets harder (the whole training set right at epoch 38 instead of 26), and the best validation loss is
clearly lower (0.3287 versus 0.3664). At the early-stopping point, all 3 seeds with dropout get more right than all 3
seeds without it: the difference lies outside the variation between seeds. But dropout doesn't prevent overfitting:
after epoch 28, the validation loss still climbs. Dropout and the L2 penalty complement each other; section 9 tries
combining them.

At inference time, `evaluate()` uses the full network, with no mask. That works thanks to the $1/(1-p)$ factor
during training. The cell below checks this on the network we just trained: it averages the output of the first
hidden layer over 200 masks, and compares that with the output without a mask.

In [ ]:
W1, b1 = drop["params"][0]
h1 = np.maximum(0, Xval[:200] @ W1 + b1)  # the first hidden layer's output for 200 validation images, inference mode
avg = np.mean([h1 * dropout_mask(h1.shape, 0.2, demo_rng) for _ in range(200)], axis=0)
rel = np.abs(avg - h1).sum() / h1.sum()
print(f"mean over 200 masks versus no mask: relative difference {pct(rel)}")
no_scale = np.mean([h1 * (demo_rng.random(h1.shape) >= 0.2) for _ in range(200)], axis=0)
print(f"if you forget to divide by 1 − p: the mean is only {pct(no_scale.sum() / h1.sum(), 1)} of the true output")

With the division by $1 - p$, the average over many masks is very close to the output of the full network (the
difference that remains comes from drawing only 200 masks). Forget the division, and during training the next layer
gets used to only 80.0% of the signal, and then at inference time each neuron is 1.25 times as large as what it got
used to. (For the second hidden layer, the average over masks only *approximates* the full network, because the ReLU
in between is nonlinear. Srivastava et al. describe using the full network at inference time as an approximation of
averaging over all the subnetworks, and it works well in practice.)

## 8 · Data augmentation: the same 1,000 images, in a different position every epoch

The best way to fight overfitting is to get more real data. Almost as good is to create “half-real” data from the
data we have, with transformations that don't change the label: a 3 shifted 2 pixels to the left is still a 3. The
function below (from `overfit.py`) shifts each image by a random whole number of pixels in $[-2, 2]$ along each axis,
filling the empty space with 0.

In [ ]:
def shift_images(x, rng):
    """Shift each image by a random whole number of pixels in [−SHIFT, SHIFT] along each axis (filling with 0)."""
    n = len(x)
    pad = np.zeros((n, 28 + 2 * SHIFT, 28 + 2 * SHIFT))
    pad[:, SHIFT : SHIFT + 28, SHIFT : SHIFT + 28] = x.reshape(n, 28, 28)
    d = rng.integers(0, 2 * SHIFT + 1, (n, 2))  # the top-left corner of the 28×28 window in the padded image
    return np.stack([pad[k, d[k, 0] : d[k, 0] + 28, d[k, 1] : d[k, 1] + 28] for k in range(n)]).reshape(n, 784)


views = shift_images(np.repeat(Xtr[:1], 8, axis=0), np.random.default_rng(4))
fig, axes = plt.subplots(1, 9, figsize=(10, 1.5))
for ax, img, title in zip(axes, [Xtr[0], *views], ["original", *[f"shift {k}" for k in range(1, 9)]]):
    ax.imshow(img.reshape(28, 28), cmap="gray_r")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.show()
print(
    f"{(2 * SHIFT + 1) ** 2} possible positions for each image; 1,000 images × {(2 * SHIFT + 1) ** 2} = {num(1000 * (2 * SHIFT + 1) ** 2, 0)} different images"
)

The figure above shows the first training image and 8 random shifts of it. An MLP doesn't know that two images 1
pixel apart are “the same picture”: to it, each position is 784 entirely different numbers. That makes memorizing
much harder. We run 150 epochs, measuring the validation set every 5 epochs:

In [ ]:
aug = train(aug=True)
check(aug, config(aug=True))
ca = config(aug=True)
for name, log, c in [("no shifts", base, config()), ("shifts ±2 px", aug, ca)]:
    print(
        f"{name:>12}: epoch 150 training {pct(log['train'][150][1] / 1000, 1)} correct, validation "
        f"{pct(log['val'][150][1] / 10000)} correct · early stopping (per the demo) keeps epoch {c['best']}: validation "
        f"loss {num(c['valLoss'][c['best']], 4)}, test {pct(c['testBest'] / 10000)} correct"
    )
print(
    f"shifts ±2 px: first gets all 1,000 training images right at epoch {first_perfect(aug)} (no shifts: epoch {first_perfect(base)})"
)
assert first_perfect(aug) == 130 and aug["val"][150][1] > 9300, "differs from the demo: check the data"

This is the biggest jump in the whole notebook: the validation accuracy at epoch 150 goes from 89.89% to 94.10%, and
the best validation loss is almost halved (0.3664 → 0.1980). The network *can* still memorize (it reaches 100% of the
training images once, at epoch 130), but it takes 5 times as many epochs, because every epoch it sees the images in
new positions. The shifts teach the network something true about handwriting that 1,000 images don't say on their
own: where a stroke sits matters less than its shape.

## 9 · Choose on the validation set, report on the test set

We now have four methods: early stopping, the L2 penalty, dropout, image shifts. What is the best way to combine them?
The demo ran the whole grid $\lambda \in \{0, 0.0003, 0.001, 0.003, 0.01\}$ × $p \in \{0, 0.2, 0.5\}$ × with/without
image shifts, 3 seeds per combination, 150 epochs, with early stopping. The four runs above are four seed-0 cells of
that grid, and they all match. The rest takes a few minutes, so we read the results the demo saved.

The selection rule: the combination with the **highest validation accuracy** at the early-stopping weights, averaged
over 3 seeds. The test set is not used to choose; we only read it at the chosen combination.

In [ ]:
def label(c):
    return f"λ = {tick(c['lam'])}, p = {tick(c['p'])}, {'with shifts' if c['aug'] else 'no shifts'}"


cfgs = runs["configs"]
chosen = max(cfgs, key=lambda c: seed_stats(c, "valOk")[0])
best_plain = max((c for c in cfgs if not c["aug"]), key=lambda c: seed_stats(c, "valOk")[0])
print(
    f"{len(cfgs)} combinations × {len(runs['seeds'])} seeds. Validation accuracy (mean, min–max over 3 seeds) → test accuracy:"
)
for name, c in [("no regularization", config()), ("best without image shifts", best_plain), ("best", chosen)]:
    print(f"  {name:>27}: {label(c):<36} validation {fmt_range(c, 'valOk')} → test {fmt_range(c, 'testOk')}")
ranked = sorted(cfgs, key=lambda c: -seed_stats(c, "valOk")[0])
second = ranked[1]
overlap = seed_stats(second, "valOk")[2] >= seed_stats(chosen, "valOk")[1]
print(f"  {'runner-up':>27}: {label(second):<36} validation {fmt_range(second, 'valOk')}; overlaps the best: {overlap}")
n_better = sum(seed_stats(c, "valOk")[0] > seed_stats(best_plain, "valOk")[0] for c in cfgs if c["aug"])
print(f"combinations with image shifts that beat the best one without: {n_better}/{sum(c['aug'] for c in cfgs)}")
for aug_on in [False, True]:
    wins = sum(
        seed_stats(config(l_, 0.5, aug_on), "valOk")[0] > seed_stats(config(l_, 0, aug_on), "valOk")[0]
        for l_ in runs["lambdas"]
    )
    c5, c0 = config(0, 0.5, aug_on), config(0, 0, aug_on)
    print(
        f"{'with' if aug_on else 'without'} image shifts: p = 0.5 beats p = 0 at {wins}/{len(runs['lambdas'])} values of λ · "
        f"λ = 0, epoch 150: training {pct(c5['trainOk'][-1] / 1000, 1)} correct (p = 0.5), {pct(c0['trainOk'][-1] / 1000, 1)} (p = 0)"
    )
full = runs["full"]
print(
    f"for comparison: the same network trained on all {num(full['nTrain'], 0)} images for {full['epochs']} epochs: test accuracy {pct(full['testAcc'])}"
)

The figure below plots the whole grid: validation accuracy (mean over 3 seeds, the bars show min–max) against
$\lambda$, without image shifts on the left and with image shifts on the right. Each $p$ has its own color, marker
shape and line style. The black star is the chosen combination.

In [ ]:
P_STYLE = {0: (GREY, "o", "-"), 0.2: (BLUE, "s", (0, (6, 3))), 0.5: (ORANGE, "^", (0, (1, 1.5)))}
lams = runs["lambdas"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, aug_on in zip(axes, [False, True]):
    for p_, (color, mk, ls) in P_STYLE.items():
        st = [seed_stats(config(lam_, p_, aug_on), "valOk") for lam_ in lams]
        m = np.array([s[0] for s in st])
        err = [m - [s[1] for s in st], [s[2] for s in st] - m]
        x = np.arange(len(lams)) + (p_ - 0.25) * 0.25  # offset slightly so the bars don't overlap
        ax.errorbar(
            x, m, yerr=err, color=color, marker=mk, ls=ls, ms=6, lw=1.5, capsize=3, label=f"dropout p = {tick(p_)}"
        )
    if chosen["aug"] == aug_on:
        k = lams.index(chosen["lam"])
        ax.plot(k + (chosen["p"] - 0.25) * 0.25, seed_stats(chosen, "valOk")[0], "*", color="black", ms=15, zorder=5)
    ax.set_xticks(range(len(lams)), [tick(v) for v in lams])
    ax.set(xlabel="L2 penalty: λ", title="With image shifts of ±2 px" if aug_on else "Without image shifts")
    ax.grid(alpha=0.3, axis="y")
axes[0].set_ylabel("validation accuracy (early stopping)")
axes[1].legend(frameon=False, fontsize=9, loc="lower left")  # on the right, whose lower left corner is empty
fmt_axes(fig)
for ax in axes:
    ax.set_xticks(range(len(lams)), [tick(v) for v in lams])  # keep the λ labels in the form 0.0003, not positions 0…4
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
plt.tight_layout()
plt.show()

The chosen combination combines all three: a light L2 penalty, dropout 0.2 and image shifts. It is also the best
combination on the test set, but we didn't use that to choose it. Three things can be read from the figure:
- **Image shifts win most clearly.** 14/15 combinations with image shifts beat the best combination without them,
  far beyond the variation between seeds. No knob on the weights achieves that.
- **Too strong hurts.** $\lambda = 0.01$ is the worst at every $p$, with or without image shifts. Dropout $p = 0.5$
  helps at 4/5 values of $\lambda$ without image shifts, but once the images are shifted it does worse than no
  dropout at every $\lambda$: the two sources of noise added together are too much for this small network, which no
  longer fits all of the training data (96.3% versus 99.9%).
- **At the top, the differences are within the variation between seeds.** The 3-seed range of the chosen combination
  overlaps the range of the runner-up, so “best” here is only slightly better than its neighbor.

A large gap remains: even with the best regularization, 1,000 images give 94.78% on the test set, while the network
trained on all 60,000 images (Lesson 04) reaches 97.27%. More real data is still the best way to fight overfitting.

## 10 · What about even bigger? Double descent

The picture “the bigger the model, the more easily it overfits” suggests a U-shaped curve: the validation error falls
as the model grows, then rises once the model is big enough to memorize. To sweep the size without training dozens of
networks, the demo uses a simpler model on the same 1,000 images (Belkin et al., 2019):

$$\phi(x) = \mathrm{ReLU}(xV), \qquad \hat y = \phi(x)\,W$$

$V$ is a **random, fixed** 784 × N matrix (each column normalized to length 1); only $W$ (N × 10) is learned, by
least squares on the one-hot labels $Y$. Let $F = \phi(X_{\text{train}})$ be the 1,000 × N feature matrix. No
gradient is needed, because there is an exact solution:
- **N < 1,000:** fewer parameters than images, so it usually can't fit them all; $W = (F^\top F)^{-1} F^\top Y$ is
  the best fit.
- **N ≥ 1,000:** infinitely many $W$ fit every image exactly; $W = F^\top (F F^\top)^{-1} Y$ is the one with the
  **smallest** $\lVert W\rVert$ (like `np.linalg.lstsq`).
- **With an L2 penalty:** $W = F^\top (F F^\top + \lambda I)^{-1} Y$ minimizes
  $\lVert FW - Y\rVert^2 + \lambda\lVert W\rVert^2$, for every N. The demo uses $\lambda = 1$.

As in `overfit.py`, N features are the first N columns of one and the same random matrix (seed 7), so a larger model
contains the smaller ones. The notebook sweeps N up to 3,000; the rest of the curve (up to N = 20,000) comes from the
demo's file, because the feature matrix of the validation set would then be too large for 2 CPU cores.

### 🤔 Predict first
The cell below prints the validation error at N = 400 (fewer parameters than images) and at N = 3,000 (three times as
many). At N = 1,000, exactly the number of training images, is the validation error between those two levels, lower
than both, or higher than both?

In [ ]:
WIDTHS = [w for w in runs["doubleDescent"]["widths"] if w <= 3000]
V_all = np.random.default_rng(7).normal(size=(784, runs["doubleDescent"]["widths"][-1]))  # the demo's random numbers
V = V_all[:, : WIDTHS[-1]] / np.linalg.norm(V_all[:, : WIDTHS[-1]], axis=0)
print(
    f"random matrix 784 × {num(V_all.shape[1], 0)} as in the demo; the notebook uses its first {num(V.shape[1], 0)} columns"
)
F_tr = np.maximum(0, Xtr @ V)  # features of the 1,000 training images; N features = the first N columns
F_val = np.maximum(0, Xval32 @ V.astype(np.float32))  # of the 10,000 validation images (float32 for speed)
Y = np.eye(10)[ytr]  # one-hot labels


def fit_last_layer(F, ridge=0.0):
    """W for ŷ = F W: least squares, the exact fit with the smallest ‖W‖, or with an L2 penalty (ridge > 0)."""
    n_img, n = F.shape
    if n < n_img and not ridge:
        return np.linalg.solve(F.T @ F, F.T @ Y)
    return F.T @ np.linalg.solve(F @ F.T + ridge * np.eye(n_img), Y)


def n_wrong(F, W, y):
    return int(((F @ W.astype(F.dtype)).argmax(1) != y).sum())


dd = {}
for n in WIDTHS:
    W, W_r = fit_last_layer(F_tr[:, :n]), fit_last_layer(F_tr[:, :n], ridge=runs["doubleDescent"]["ridge"])
    errs = [n_wrong(F[:, :n], w, y) for w in (W, W_r) for F, y in ((F_tr, ytr), (F_val, yval))]
    dd[n] = {"err": errs, "norm": float(np.linalg.norm(W))}
for n in [100, 400, 900, 1000, 1100, 3000]:
    tr, va, _, va_r = dd[n]["err"]
    print(
        f"N = {num(n, 0):>5}: training error {pct(tr / 1000, 1):>5} · validation error {pct(va / 10000, 1):>5} · "
        f"‖W‖ = {num(dd[n]['norm'], 1):>7} · with the L2 penalty: validation error {pct(va_r / 10000, 1)}"
    )
ref = runs["doubleDescent"]
for k, n in enumerate(WIDTHS):
    want = [ref[m][e][k] for m in ("minNorm", "withRidge") for e in ("trainErr", "valErr")]
    assert max(abs(a - b) for a, b in zip(dd[n]["err"], want)) <= 3, f"differs from the demo at N = {n}"
peak = dd[1000]["err"][1]
assert peak > dd[400]["err"][1] and peak > dd[3000]["err"][1] and dd[1000]["err"][3] < peak
print("matches the demo at every N ≤ 3,000")

Higher than both. The figure below plots the whole curve, with N on a log scale: the validation error without a
penalty (orange, dashed, squares), with an L2 penalty $\lambda = 1$ (green, solid, triangles), and the training error
without a penalty (blue, dotted, circles). Filled markers are computed by the notebook (N ≤ 3,000); hollow ones come
from the demo's file. The vertical grey line is N = 1,000, exactly the number of training images.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
N_all = np.array(ref["widths"])
series = [
    ("validation error, no penalty", ref["minNorm"]["valErr"], 10000, 1, ORANGE, "s", (0, (6, 3))),
    ("validation error, L2 penalty (λ = 1)", ref["withRidge"]["valErr"], 10000, 3, GREEN, "^", "-"),
    ("training error, no penalty", ref["minNorm"]["trainErr"], 1000, 0, BLUE, "o", (0, (1, 1.5))),
]
for name, stored, total, k, color, mk, ls in series:
    ax.plot(N_all, np.array(stored) / total, color=color, ls=ls, lw=1.6, label=name)
    live = np.array([dd[n]["err"][k] for n in WIDTHS]) / total
    ax.plot(WIDTHS, live, mk, color=color, ms=6)
    far = N_all > WIDTHS[-1]
    ax.plot(N_all[far], np.array(stored)[far] / total, mk, color=color, mfc="white", ms=6)
ax.axvline(len(Xtr), color=GREY, ls=":", lw=1.2)
ax.annotate("N = number of training images", (len(Xtr), 0.95), xytext=(6, 0), textcoords="offset points", fontsize=9)
ax.set(xscale="log", xlabel="number of random features N (log scale)", ylabel="error rate", ylim=(0, 1))
ax.legend(frameon=False, fontsize=9, loc="upper right", bbox_to_anchor=(1, 0.9))
ax.grid(alpha=0.3, which="both")
fmt_axes(fig)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
plt.show()
classic = min(
    (n for n in ref["widths"] if n < len(Xtr)), key=lambda n: ref["minNorm"]["valErr"][ref["widths"].index(n)]
)
k_c, k_last = ref["widths"].index(classic), len(ref["widths"]) - 1
print(f"best before the peak: N = {num(classic, 0)}, validation error {pct(ref['minNorm']['valErr'][k_c] / 10000, 1)}")
print(
    f"peak: N = 1,000, {pct(dd[1000]['err'][1] / 10000, 1)} · ‖W‖ = {num(dd[1000]['norm'], 1)} (N = 3,000: {num(dd[3000]['norm'], 1)})"
)
print(
    f"N = {num(ref['widths'][-1], 0)} (the demo's file): no penalty {pct(ref['minNorm']['valErr'][k_last] / 10000, 1)}, "
    f"L2 penalty {pct(ref['withRidge']['valErr'][k_last] / 10000, 1)}"
)

The curve falls twice, hence the name **double descent** (Belkin et al., 2019):
- **The classical regime (N < 1,000):** the validation error falls and then rises, the familiar U shape; it is lowest
  at N = 400.
- **The peak (N = 1,000):** the last layer has just enough parameters to fit every image exactly, and there is only
  *one* way to fit them. The matrix $F$ is nearly singular, so that one fit needs very large weights
  ($\lVert W\rVert$ = 1,052.9 instead of a few dozen), and on new images it is 87.9% wrong, worse even than random
  guessing.
- **The overparameterized regime (N > 1,000):** there are infinitely many exact fits, and the solution with the
  smallest $\lVert W\rVert$ gets ever “smoother”. The validation error falls a second time, below even the best point
  of the classical regime.

A moderate L2 penalty ($\lambda = 1$) makes the peak almost disappear: it doesn't let $W$ grow large, so no size is a
“disaster”. Nakkiran et al. (2020) proved this for some linear regression models: an optimally tuned L2 penalty makes
the validation error decrease steadily with the size of the model.

With a full MLP like the one in the sections above, this peak is usually much weaker, and clearest when the labels are
noisy (Nakkiran et al., 2019). The thing to remember is not “a bigger model is always better”, but this: *the number
of parameters relative to the amount of data* does not by itself decide overfitting; what matters is which of the
infinitely many exact fits training picks.

## 11 · Try changing things

The cells below rerun the same functions with different parameters. You can edit the values and run them again.

### Does regularization prevent memorizing meaningless labels?
Zhang et al. found that weight decay and dropout don't prevent a network from memorizing random labels. The cell
below tries image shifts on our network, 150 epochs, shuffled labels. Add `("L2 penalty", {"lam": 3e-3})` or
`("dropout", {"p": 0.2})` to the list to try the other two (each run takes a few seconds).

In [ ]:
for name, kw in [("image shifts ±2 px", {"aug": True})]:
    log = train(labels=SHUFFLED, val_every=EPOCHS, **kw)
    first = first_perfect(log)
    print(
        f"shuffled labels + {name:<18}: first gets 100% of the training images right: {first if first is not None else 'not yet'} · "
        f"epoch 150: training {pct(log['train'][150][1] / 1000, 1)} correct, validation {pct(log['val'][150][1] / 10000)} correct"
    )

Image shifts make memorizing much harder: after 150 epochs the network has memorized only 38.5% of the meaningless
labels (without shifts: all of them by epoch 68), because every epoch it has to remember a new set of images. Even so,
validation stays at the level of random guessing: regularization doesn't turn noise into a rule.

### Early stopping's patience
The demo waits 10 epochs. Waiting less stops earlier and costs less, but makes it easier to stop in a temporary dip.
The cell below recomputes early stopping on the validation curves (measured every epoch) of all 30 seed-0
combinations in the demo's file, with other patience values:

In [ ]:
def early_stop(val_loss, patience):
    best = 0
    for e in range(1, len(val_loss)):
        if val_loss[e] < val_loss[best]:
            best = e
        elif e - best >= patience:
            return best, e
    return best, len(val_loss) - 1


for patience in [3, 5, 10, 20, 40]:
    picks = [early_stop(c["valLoss"], patience) for c in cfgs]
    print(
        f"patience {patience:>2} epochs: stops on average at epoch {num(np.mean([s for _, s in picks]), 0):>3} · "
        f"validation accuracy at the kept weights: {pct(np.mean([c['valOk'][b] / 10000 for c, (b, _) in zip(cfgs, picks)]))}"
    )

Waiting 3 epochs is the cheapest but keeps clearly worse weights; from 10 epochs on, the extra gain is very small.
This too is a hyperparameter, and it too is chosen on the validation set.

### A different random feature matrix
Is the peak at N = 1,000 just the luck of seed 7? Change the seed of $V$ and recompute the validation error at
N = 400, 1,000 and 1,100:

In [ ]:
print(f"seed 7 (section 10): {' · '.join(pct(dd[n]['err'][1] / 10000, 1) for n in (400, 1000, 1100))}")
for v_seed in [8, 9]:
    Vs = np.random.default_rng(v_seed).normal(size=(784, 20000))[
        :, :1100
    ]  # the first 1,100 columns, drawn as in the demo
    Vs /= np.linalg.norm(Vs, axis=0)
    Fs_tr, Fs_val = np.maximum(0, Xtr @ Vs), np.maximum(0, Xval32 @ Vs.astype(np.float32))
    errs = [n_wrong(Fs_val[:, :n], fit_last_layer(Fs_tr[:, :n]), yval) / 10000 for n in (400, 1000, 1100)]
    print(f"seed {v_seed}: {' · '.join(pct(e, 1) for e in errs)}")

With all three random matrices, the validation error at N = 1,000 is many times higher than on either side: the peak
is caused by N = the number of images, not by the luck of one seed.

## 12 · Limits, and the bridge to the next lesson

Everything here ran on a shallow network: two hidden layers, and the network *can learn* easily. Its only problem is
that it learns too well. When a network is much deeper, a different problem appears even before overfitting: the
network can't learn. The signal and the gradient passing through dozens of layers can shrink toward 0 or grow without
bound, and then no L2 penalty or dropout can help. We have also used He initialization since the start of the course
without explaining why. Both are the subject of Lesson 09.

A note about large language models: they usually read each text only once or a few times, so they have little chance
to memorize. That is why many pretrained models do **not** use dropout: PaLM notes that the model “was trained without
dropout, although dropout of 0.1 is used for finetuning in most cases” (Chowdhery et al., 2022, section 5). The first
GPT still used dropout 0.1 (Radford et al., 2018). Weight decay, on the other hand, is still common, for example
LLaMA's AdamW with $\lambda = 0.1$ (Lesson 07).

## Summary
- On 1,000 images, the 109,386-parameter network gets the whole training set right from epoch 26; the validation loss
  is lowest at epoch 20 and then rises 30% by epoch 150, while the validation accuracy barely moves: the network is
  wrong *more confidently*, not more often.
- With randomly shuffled labels, the network still memorizes all 1,000 meaningless pairs (epoch 68), and guesses at
  random on new images: a low training loss doesn't prove that the network has understood.
- The L2 penalty $\lambda\lVert W\rVert^2$ with plain SGD is weight decay: each step multiplies $W$ by
  $1 - 2\eta\lambda$. With $\lambda = 0.003$, ‖W‖² falls to 95.1 instead of rising to 495, and the validation loss no
  longer climbs.
- Dropout randomly switches off neurons, with one mask per image and a new mask at every step, and divides by $1 - p$
  so that the full network can be used at inference time. It doesn't make the weights smaller, but it makes
  memorizing harder.
- Image shifts of ±2 px help more than any combination of λ and p: the validation accuracy at epoch 150 goes from
  89.89% to 94.10%. Chosen on the validation set, the best combination (λ = 0.001, p = 0.2, image shifts) reaches
  94.78% on the test set, still far below the 97.27% of 60,000 images.
- Double descent: with random features, the validation error shoots up to 87.9% right at N = the number of images,
  then falls a second time as N grows; a moderate L2 penalty erases almost all of the peak.

## Review questions
1. The validation loss rises by 30% from epoch 20 to epoch 150, yet the validation accuracy doesn't fall. What
   changed?
2. Why must $\lambda$, $p$ and the stopping epoch be chosen on the validation set, and not on the test set?
3. With plain SGD, write one update step with the L2 penalty as “shrink, then step”. What is the shrink factor? Why
   is this no longer the case with Adam?
4. At inference time, why can we use the full network without adjusting the weights? What happens if you forget to
   divide by $1 - p$ during training?
5. Dropout makes ‖W‖² larger, the L2 penalty makes ‖W‖² smaller, and both fight overfitting. So is “small weights”
   the only explanation for good generalization?
6. In the random-features experiment, why is the validation error highest exactly at N = 1,000, and why does it fall
   again as N keeps growing?

<details><summary>Hints and answers</summary>

1. The network became more confident even on the images it gets wrong: the mean loss on the wrong images rose from
   2.85 to 4.41, and the number of wrong images with ≥ 99% confidence rose from 21 to 132. Cross-entropy punishes
   that kind of error heavily; accuracy doesn't see it.
2. If you choose on the test set, the number you report is also a number that was optimized, so it is more optimistic
   than reality. The test set can only be trusted when it hasn't influenced any choice.
3. $W \leftarrow (1 - 2\eta\lambda)W - \eta\nabla\mathcal{L}$; with $\eta = 0.1$ and $\lambda = 0.003$ that is
   0.9994 per step. Adam divides both the gradient and the $2\lambda W$ term by $\sqrt{\hat v}$, so weights with
   large gradients get shrunk less (Lesson 07); AdamW takes the shrinking out of that division.
4. Thanks to the $1/(1-p)$ factor during training, the expected value of each neuron equals its value without a
   mask. Forget the division, and the next layer gets used to a signal only 80% as large (with $p = 0.2$), then
   receives a signal 1.25 times as large at inference time.
5. No. Dropout fights memorization in a different way: each neuron has to be useful in a great many subnetworks.
   Image shifts also make ‖W‖² larger, and yet they help the most. Zhang et al. concluded that explicit
   regularization is “neither necessary nor by itself sufficient” to explain generalization.
6. At N = 1,000 there is only one way to fit every image exactly, and because $F$ is nearly singular, that way needs
   a very large $\lVert W\rVert$. When N is larger, there are infinitely many ways to fit, and the solution with the
   smallest $\lVert W\rVert$ gets ever smoother.
</details>

## Further reading
- G. Hinton et al. (2012), [Improving neural networks by preventing co-adaptation of feature
  detectors](https://arxiv.org/abs/1207.0580): the first paper on dropout.
- N. Srivastava et al. (2014), [Dropout: A Simple Way to Prevent Neural Networks from
  Overfitting](https://jmlr.org/papers/v15/srivastava14a.html): there $p$ is the probability of *keeping* a unit,
  usually 0.5–0.8 for hidden layers.
- C. Zhang et al. (2016), [Understanding deep learning requires rethinking generalization](https://arxiv.org/abs/1611.03530):
  the random-label experiment.
- M. Belkin et al. (2019), [Reconciling modern machine learning practice and the bias-variance
  trade-off](https://arxiv.org/abs/1812.11118): double descent with random features.
- P. Nakkiran et al. (2019), [Deep Double Descent](https://arxiv.org/abs/1912.02292), and (2020)
  [Optimal Regularization Can Mitigate Double Descent](https://arxiv.org/abs/2003.01897).
- A. Karpathy (2019), [A Recipe for Training Neural Networks](https://karpathy.github.io/2019/04/25/recipe/), the
  “Regularize” section: more real data first, then data augmentation, dropout, weight decay, early stopping.
- [Dive into Deep Learning, section 5.6 · Dropout](https://d2l.ai/chapter_multilayer-perceptrons/dropout.html).
- [MLU-Explain · Double Descent](https://mlu-explain.github.io/double-descent/) (interactive).
- A. Chowdhery et al. (2022), [PaLM](https://arxiv.org/abs/2204.02311), section 5; A. Radford et al. (2018),
  [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf).

**Next lesson:** [Lesson 09 · Vanishing gradients & initialization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/vanishing-gradient/):
when a network is dozens of layers deep, the signal and the gradient fade away or blow up, and initializing the right
way keeps them stable.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")